# Dubai Property Price Estimator

### Context

The project aims to estimate the fair **asking price** of properties listed in Dubai. Each listing has features such as location, bedrooms, bathrooms, and property type, and a target **price**, that the model learns to predict. Sellers can use the estimated price to check whether a listing is **underpriced** or **overpriced**.

### Content of Dataset

Column 1 : **Title** represents the headline of the advertisement for this property   
Column 2 : **displayAddress** represents the location of the listing **(key feature since some locations are prime thus higher price)**  
Column 3 : **bathrooms** represents the number of bathrooms in the property **(key feature since more bathrooms means higher price )**  
Column 4 : **bedrooms** represents the number of bedrooms in the property **(key feature since more bedrooms means higher price)**  
Column 5 : **addOn** represents the date the listing was posted  
Column 6 : **type** represents the type of listing (residential or commercial)  
Column 7 : **price** represents the asking price of the property in **AED** **(Target)**  
Column 8 : **verified** represents whether the listing was verified by the portal  
Column 9 : **priceDuration** represents the time period the price covers (eg rental listings show monthly or yearly, since the data only contains properties being sold, it contains 'sell' only)  
Column 10 : **sizeMin** represents the size of the property in **square feet** **(Key feature since bigger properties mean bigger price)**  
Column 11 : **furnishing** represents whether the property is furnished, not furnished, or partly furnished



### Project Goals:

As a **supervised** regression problem, the goal of the project is:  
- Predict the asking price of UAE residential properties for sale from location, size, bedrooms, bathrooms, and furnishing.
- Explore the data to understand what drives price.
- Compare regression models and select the best one.
- Deploy the model as a web service with Docker so sellers can check whether a price is fair.


### Data Preperation:

Dataset from https://www.kaggle.com/datasets/kanchana1990/uae-real-estate-2024-dataset/data 

#### Understanding the Data

In [143]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt

In [77]:
df = pd.read_csv('Data/uae_real_estate_2024.csv')

In [78]:
df.head()

,title,displayAddress,bathrooms,bedrooms,addedOn,type,price,verified,priceDuration,sizeMin,furnishing,description
0,Great ROI I High Floor I Creek View,"Binghatti Canal, Business Bay, Dubai",3,2,2024-08-14T12:02:53Z,Residential for Sale,2500000,True,sell,1323 sqft,NO,MNA Properties is delighted to offer this apar...
1,Full Sea View | Beach Life | Brand New Residence,"La Vie, Jumeirah Beach Residence, Dubai",3,2,2024-08-13T05:45:16Z,Residential for Sale,7200000,True,sell,1410 sqft,YES,"Apartment for sale in La Vie, Jumeirah Beach R..."
2,Green Belt | Corner Unit | Spacious Plot,"La Rosa 6, Villanova, Dubai Land, Dubai",3,4,2024-08-14T06:24:28Z,Residential for Sale,3600000,True,sell,2324 sqft,NO,Treo Homes is very pleased to be bringing to t...
3,2BR+Study | Near Pool and Park | Private,"Springs 15, The Springs, Dubai",3,2,2024-08-15T06:07:22Z,Residential for Sale,2999999,True,sell,1647 sqft,NO,2 Bedrooms + Study | Near Pool &amp; Park | Pr...
4,Vacant | Well Maintained | Area Expert,"Noor Townhouses, Town Square, Dubai",3,3,2024-08-09T08:28:59Z,Residential for Sale,2449999,True,sell,2105 sqft,NO,-Type 1\n-3 Bed+Maid\n-Close To Amenities\n-BU...


In [79]:
print(f'The dataset has {df.shape[0]} rows and {df.shape[1]} columns')

The dataset has 5058 rows and 12 columns


In [80]:
df.sample(5) #Picks randomly 5 rows (more data can be seen )

,title,displayAddress,bathrooms,bedrooms,addedOn,type,price,verified,priceDuration,sizeMin,furnishing,description
170,Luxurious | Panoramic views | Newly Renovated,"Building 19, City Walk, Dubai",4,2,2024-07-27T13:46:46Z,Residential for Sale,6000000,True,sell,1752 sqft,YES,Discover luxurious living in this completely u...
1888,Full Canal View I High Floor I Vacant soon,"Bay's Edge, Business Bay, Dubai",3,2,2024-08-05T08:30:57Z,Residential for Sale,1899000,True,sell,1254 sqft,NO,Reloc8 is thrilled to showcase this stunning p...
3764,Ready To Move 2-BDR Townhouse In Masaar Freehold,"Sendian, Masaar, Tilal City, Sharjah",4,2,2024-07-23T09:15:22Z,Residential for Sale,1999990,False,sell,1850 sqft,NO,An opportunity to own a townhouse with 2 maste...
2372,Near Metro/Brand New/Amazing Terrace/Exclusive,"Zazen Gardens, Al Furjan, Dubai",2,1,2024-06-21T10:51:14Z,Residential for Sale,1400000,True,sell,917 sqft,NO,DESEAR REAL ESTATE is delighted to offer this ...
593,EXPERIENCE LUXURY | STUDIO,"Diamondz By Danube, Jumeirah Lake Towers, Dubai",1,studio,2024-07-20T13:13:49Z,Residential for Sale,1100000,False,sell,420 sqft,PARTLY,Introducing the latest pinnacle of luxury livi...


In [81]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5058 entries, 0 to 5057
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   title           5058 non-null   str  
 1   displayAddress  5058 non-null   str  
 2   bathrooms       4938 non-null   str  
 3   bedrooms        4935 non-null   str  
 4   addedOn         5058 non-null   str  
 5   type            5058 non-null   str  
 6   price           5058 non-null   int64
 7   verified        5058 non-null   bool 
 8   priceDuration   5058 non-null   str  
 9   sizeMin         5058 non-null   str  
 10  furnishing      5056 non-null   str  
 11  description     5057 non-null   str  
dtypes: bool(1), int64(1), str(10)
memory usage: 439.7 KB


In [82]:
df.dtypes[df.dtypes == 'str'].index # finds all columns that are strings 

Index(['title', 'displayAddress', 'bathrooms', 'bedrooms', 'addedOn', 'type',
       'priceDuration', 'sizeMin', 'furnishing', 'description'],
      dtype='str')

We will need to clean the data since most numerical features are strings therefore we will have to change them back

In [83]:
df.dtypes

title               str
displayAddress      str
bathrooms           str
bedrooms            str
addedOn             str
type                str
price             int64
verified           bool
priceDuration       str
sizeMin             str
furnishing          str
description         str
dtype: object

In [84]:
df.isna().sum() #Since the values that are missing less than 50% for all the column we can fill them rather than removing them  

title               0
displayAddress      0
bathrooms         120
bedrooms          123
addedOn             0
type                0
price               0
verified            0
priceDuration       0
sizeMin             0
furnishing          2
description         1
dtype: int64

In [85]:
df.describe()

,price
count,5.058000e+03
mean,5.050924e+06
std,1.000928e+07
min,1.000000e+05
25%,1.200000e+06
50%,2.350000e+06
75%,4.499375e+06
max,1.990000e+08


Are there any duplicate observations?

In [86]:
df.duplicated(keep ='first').sum() 

np.int64(0)

There are 0 duplicated columns in the dataset 

### Drop features priceDuration and type 

Since the features priceDuration and type are always the same having them won't impact the model.The feature type is always 'residential for sale' and the priceDuration is always 'year' thus they are not impacting price 

In [87]:
df['priceDuration'].unique(), df['type'].unique()

(<StringArray>
 ['sell']
 Length: 1, dtype: str,
 <StringArray>
 ['Residential for Sale']
 Length: 1, dtype: str)

In [88]:
df = df.drop(['priceDuration','type'], axis = 1)

In [89]:
df 

,title,displayAddress,bathrooms,bedrooms,addedOn,price,verified,sizeMin,furnishing,description
0,Great ROI I High Floor I Creek View,"Binghatti Canal, Business Bay, Dubai",3,2,2024-08-14T12:02:53Z,2500000,True,1323 sqft,NO,MNA Properties is delighted to offer this apar...
1,Full Sea View | Beach Life | Brand New Residence,"La Vie, Jumeirah Beach Residence, Dubai",3,2,2024-08-13T05:45:16Z,7200000,True,1410 sqft,YES,"Apartment for sale in La Vie, Jumeirah Beach R..."
2,Green Belt | Corner Unit | Spacious Plot,"La Rosa 6, Villanova, Dubai Land, Dubai",3,4,2024-08-14T06:24:28Z,3600000,True,2324 sqft,NO,Treo Homes is very pleased to be bringing to t...
3,2BR+Study | Near Pool and Park | Private,"Springs 15, The Springs, Dubai",3,2,2024-08-15T06:07:22Z,2999999,True,1647 sqft,NO,2 Bedrooms + Study | Near Pool &amp; Park | Pr...
4,Vacant | Well Maintained | Area Expert,"Noor Townhouses, Town Square, Dubai",3,3,2024-08-09T08:28:59Z,2449999,True,2105 sqft,NO,-Type 1\n-3 Bed+Maid\n-Close To Amenities\n-BU...
...,...,...,...,...,...,...,...,...,...,...
5053,07 Layout | Handover Q2 2025 | Multiple Units ...,"Sobha Creek Vistas Grande, Sobha Hartland, Moh...",3,2,2024-07-10T14:07:57Z,2345000,True,1268 sqft,PARTLY,Nobel Victory Real Estate are proud to present...
5054,Emaar | Waterfront Apartment | 10% DP | 5-Yr PP,"Al Marjan Island, Ras Al Khaimah",2,1,2024-08-12T13:08:33Z,3007888,False,854 sqft,NO,"Discover Address Residence by Emaar, nestled o..."
5055,Upgraded | Spacious With Private Garden,"Oia Residence, Motor City, Dubai",5,4,2024-04-20T08:43:16Z,4000000,True,3813 sqft,NO,Rocky Real Estate presents an Exclusive and up...
5056,"10 Series | Full Sea, Burj Khalifa View Exclusive","The Grand, Dubai Creek Harbour (The Lagoons), ...",2,2,2024-07-08T07:05:56Z,3800000,True,1300 sqft,NO,Dingstone Real Estate is pleased to present to...


#### Data cleaning and Feature Engineering

In [90]:
df['sizeMin'].head()

0    1323 sqft
1    1410 sqft
2    2324 sqft
3    1647 sqft
4    2105 sqft
Name: sizeMin, dtype: str

In [91]:
df['size_sqft'] = df['sizeMin'].str.replace('sqft','').str.strip() 

In [92]:
df['size_sqft'] = df['size_sqft'].astype('float64')

In [93]:
df['size_sqft']

0       1323.0
1       1410.0
2       2324.0
3       1647.0
4       2105.0
         ...  
5053    1268.0
5054     854.0
5055    3813.0
5056    1300.0
5057    3685.0
Name: size_sqft, Length: 5058, dtype: float64

In [94]:
df.head()

,title,displayAddress,bathrooms,bedrooms,addedOn,price,verified,sizeMin,furnishing,description,size_sqft
0,Great ROI I High Floor I Creek View,"Binghatti Canal, Business Bay, Dubai",3,2,2024-08-14T12:02:53Z,2500000,True,1323 sqft,NO,MNA Properties is delighted to offer this apar...,1323.0
1,Full Sea View | Beach Life | Brand New Residence,"La Vie, Jumeirah Beach Residence, Dubai",3,2,2024-08-13T05:45:16Z,7200000,True,1410 sqft,YES,"Apartment for sale in La Vie, Jumeirah Beach R...",1410.0
2,Green Belt | Corner Unit | Spacious Plot,"La Rosa 6, Villanova, Dubai Land, Dubai",3,4,2024-08-14T06:24:28Z,3600000,True,2324 sqft,NO,Treo Homes is very pleased to be bringing to t...,2324.0
3,2BR+Study | Near Pool and Park | Private,"Springs 15, The Springs, Dubai",3,2,2024-08-15T06:07:22Z,2999999,True,1647 sqft,NO,2 Bedrooms + Study | Near Pool &amp; Park | Pr...,1647.0
4,Vacant | Well Maintained | Area Expert,"Noor Townhouses, Town Square, Dubai",3,3,2024-08-09T08:28:59Z,2449999,True,2105 sqft,NO,-Type 1\n-3 Bed+Maid\n-Close To Amenities\n-BU...,2105.0


In [95]:
df['bathrooms'].value_counts()

bathrooms
2       1209
1       1038
3        902
4        686
5        511
6        273
7        206
7+       112
none       1
Name: count, dtype: int64

In [96]:
df['bathrooms'] = df['bathrooms'].replace({'7+':'7','none': np.nan})

In [97]:
df['bathrooms'].value_counts()

bathrooms
2    1209
1    1038
3     902
4     686
5     511
7     318
6     273
Name: count, dtype: int64

In [98]:
df['bathrooms'] = df['bathrooms'].astype('float64')

In [99]:
df['bathrooms'].value_counts(dropna = False)

bathrooms
2.0    1209
1.0    1038
3.0     902
4.0     686
5.0     511
7.0     318
6.0     273
NaN     121
Name: count, dtype: int64

Bathroom feature cleanup:
It was stored as text in the data and contained two non numerical values ( **'7+'** and **'none'**)  

- **7+** (112 listings): replaced with **7**. This treats "7 or more" as 7, a small simplification that merges these listings with the existing 7-bathroom group.
- **none** (1 listing): replaced with **NaN**. A property with no bathroom is most likely missing data, so it is handled together with the other missing values.

After this step the column can be converted to a numeric type, and the missing values (about 121) can be filled later.

In [100]:
df['bedrooms'].value_counts()

bedrooms
1         1265
2         1088
3          799
4          714
studio     541
5          366
6          122
7           25
7+          15
Name: count, dtype: int64

In [101]:
df['bedrooms'] = df['bedrooms'].replace({'studio':'0', '7+':'7'})

In [102]:
df['bedrooms'].isna().sum()

np.int64(123)

In [103]:
df['bedrooms'] = df['bedrooms'].astype('float64')

In [104]:
df.sample(5)

,title,displayAddress,bathrooms,bedrooms,addedOn,price,verified,sizeMin,furnishing,description,size_sqft
4558,Exclusive | G+1 | Spacious 3BHK TH | Rented,"Creek Gate Tower 2, Creek Gate, Dubai Creek Ha...",4.0,3.0,2024-07-06T06:34:29Z,5500000,True,2161 sqft,NO,This townhouse truly embodies a lifestyle of l...,2161.0
3625,Luxurious Unit with Balcony | Community View,"Tower A, DAMAC Towers by Paramount, Business B...",3.0,3.0,2024-06-26T13:45:47Z,2700000,True,1746 sqft,YES,"Brought to you by Driven Properties, this 3 Be...",1746.0
4455,Sea View| Motivated Seller | Large Layout,"Grand Bleu Tower 2, EMAAR Beachfront, Dubai Ha...",1.0,1.0,2024-07-04T07:15:54Z,2500000,True,837 sqft,PARTLY,Billionaire Homes Real Estate offers a luxurio...,837.0
3294,Cheapest Price | Type A | Non Wire | D1 West,"District One West Phase I, District One, Moham...",5.0,4.0,2024-08-16T06:47:22Z,10800000,False,6800 sqft,NO,GPG Global Real Estate is pleased to offer you...,6800.0
4021,2 BHK in TIRAZ al jada NEW BRAND READY,"Tiraz, Naseej District, Aljada, Sharjah",3.0,2.0,2024-08-13T10:13:58Z,1240000,False,1155 sqft,PARTLY,"Apartment for sale in Tiraz, Naseej District\n...",1155.0


Bedrooms feature cleanup:
It was stored as text in the data and contained two non numerical values ( **'studio'** and **'7+'**)  

- **7+** (15 listings): replaced with **7**. This treats "7 or more" as 7, a small simplification that merges these listings with the existing 7-bedroom group.
- **studio** (541 listings): replaced with **0**. A studio property has zero seperate bedrooms thus replaced it with 0.

After this step the column can be converted to a numeric type (float) since the data had NaN values, and the missing values (about 123) can be filled later.

In [105]:
df['furnishing'] = df['furnishing'].str.capitalize()

In [106]:
df['furnishing'].unique()

<StringArray>
['No', 'Yes', 'Partly', nan]
Length: 4, dtype: str

In [107]:
df[['sizeMin', 'size_sqft']].sample(5)

,sizeMin,size_sqft
3570,2203 sqft,2203.0
4552,526 sqft,526.0
936,818 sqft,818.0
4001,1967 sqft,1967.0
2925,719 sqft,719.0


In [108]:
df['size_sqft'].isna().sum()

np.int64(0)

In [109]:
df = df.drop(['sizeMin'], axis = 1 )

Since feature sizeMin contained text (eg 1800 sqft), we just made the value numerical and dropped the sqft unit to make size_sqft and dropped the old one since they represent the same information.

In [110]:
df.head(5)

,title,displayAddress,bathrooms,bedrooms,addedOn,price,verified,furnishing,description,size_sqft
0,Great ROI I High Floor I Creek View,"Binghatti Canal, Business Bay, Dubai",3.0,2.0,2024-08-14T12:02:53Z,2500000,True,No,MNA Properties is delighted to offer this apar...,1323.0
1,Full Sea View | Beach Life | Brand New Residence,"La Vie, Jumeirah Beach Residence, Dubai",3.0,2.0,2024-08-13T05:45:16Z,7200000,True,Yes,"Apartment for sale in La Vie, Jumeirah Beach R...",1410.0
2,Green Belt | Corner Unit | Spacious Plot,"La Rosa 6, Villanova, Dubai Land, Dubai",3.0,4.0,2024-08-14T06:24:28Z,3600000,True,No,Treo Homes is very pleased to be bringing to t...,2324.0
3,2BR+Study | Near Pool and Park | Private,"Springs 15, The Springs, Dubai",3.0,2.0,2024-08-15T06:07:22Z,2999999,True,No,2 Bedrooms + Study | Near Pool &amp; Park | Pr...,1647.0
4,Vacant | Well Maintained | Area Expert,"Noor Townhouses, Town Square, Dubai",3.0,3.0,2024-08-09T08:28:59Z,2449999,True,No,-Type 1\n-3 Bed+Maid\n-Close To Amenities\n-BU...,2105.0


## Feature Engineering 

### Extracting 'city' and 'community' 


'displayAddress' has many unique values thus can't be used directly. However it follows the pattern 'building,community, city' therefore it was split into two important features city (last part) and the community (middle part)

In [111]:
def find_city(details):
    details = details.split(',')
    city = details[-1].strip()
    return city

In [112]:
df['City'] = df['displayAddress'].apply(find_city)

In [113]:
df.sample(5)

,title,displayAddress,bathrooms,bedrooms,addedOn,price,verified,furnishing,description,size_sqft,City
2786,Full Sea View | High Floor | Beach Access,"Rixos, Dubai Islands, Deira, Dubai",2.0,2.0,2024-08-06T13:02:42Z,4800000,True,No,GPG Global Real Estate is presenting 2 Bedroom...,1614.0,Dubai
610,Prime location | Leisure | Spacious,"Sonate Residences, Jumeirah Village Triangle, ...",3.0,2.0,2024-05-01T10:39:34Z,1747665,False,Yes,Luxurious community with easy access to leisur...,1307.0,Dubai
2478,Sharjah Midtown | Smart Home | 5% Down Payment,"Nesba, Aljada, Sharjah",3.0,2.0,2024-07-20T08:38:24Z,1616000,False,No,0395555942\n\n- Prime location and connectivit...,1475.0,Sharjah
4465,Villa for sale in Al Yasmine ajman,"Al Yasmeen 1, Al Yasmeen, Ajman",7.0,5.0,2024-08-05T06:05:53Z,1350000,False,No,"Receive your villa without a down payment, Al ...",3014.0,Ajman
141,Stunning Resale New 1 BR |Maryam Island| Sha...,"Rimal Residences, Maryam Island, Sharjah",1.0,1.0,2024-03-14T13:17:42Z,780000,False,No,luxury living that further aims to connect res...,650.0,Sharjah


In [114]:
def find_community(details):
    details = details.split(',')
    community = details[1].strip()
    return community

In [115]:
df['Community'] = df['displayAddress'].apply(find_community)

In [116]:
df.sample(5)

,title,displayAddress,bathrooms,bedrooms,addedOn,price,verified,furnishing,description,size_sqft,City,Community
4865,Spacious Land | Ideal Location | Best Community,"Madinat Zayed, Abu Dhabi",NaN,NaN,2024-07-05T12:23:06Z,4500000,False,No,Experience the perfect fusion of modernity and...,24291.0,Abu Dhabi,Abu Dhabi
3822,Price Dropped - Gorgeous Pool View - Upgraded ...,"Bayti Townhouses, Al Hamra Village, Ras Al Kha...",4.0,4.0,2023-06-14T05:47:41Z,2850000,False,Yes,<strong>The Property<br/><br/></strong>Al Hamr...,2668.0,Ras Al Khaimah,Al Hamra Village
2809,LUXURIOUS BRAND NEW PREMIUM LOCATION,"AZIZI Riviera, Meydan One, Meydan, Dubai",1.0,0.0,2024-05-29T10:46:51Z,835000,True,No,Tamani Real Estate proudly presents this Elega...,346.0,Dubai,Meydan One
4790,3BR Smart Home | saving 50% SEWA | SMART HOME,"Sharjah Sustainable City, Sharjah",4.0,3.0,2024-01-02T16:34:40Z,1700000,False,No,FREEHOLD – OPEN TO ALL NATIONALITIES\n\nReady ...,2015.0,Sharjah,Sharjah
5001,FULL FLOOR AND UNIT WISE SALE | CLASSIC DECOR,"Rahaal 1, Madinat Jumeirah Living, Umm Suqeim,...",2.0,1.0,2024-06-17T10:33:58Z,2600000,False,No,Savos Realty LLC proudly presents this contemp...,835.0,Dubai,Madinat Jumeirah Living


In [117]:
df['City'].value_counts(dropna = False) 

City
Dubai             4031
Abu Dhabi          370
Sharjah            267
Ras Al Khaimah     201
Ajman              170
Umm Al Quwain       11
Fujairah             4
Al Ain               4
Name: count, dtype: int64

In [118]:
df['Community'].nunique()

530

In [120]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5058 entries, 0 to 5057
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   title           5058 non-null   str    
 1   displayAddress  5058 non-null   str    
 2   bathrooms       4937 non-null   float64
 3   bedrooms        4935 non-null   float64
 4   addedOn         5058 non-null   str    
 5   price           5058 non-null   int64  
 6   verified        5058 non-null   bool   
 7   furnishing      5056 non-null   str    
 8   description     5057 non-null   str    
 9   size_sqft       5058 non-null   float64
 10  City            5058 non-null   str    
 11  Community       5058 non-null   str    
dtypes: bool(1), float64(3), int64(1), str(7)
memory usage: 439.7 KB


### Dropping unused columns 

- 'displayAddress' : replaced by extracted features like 'Community' and 'City'  
- 'addedOn' : Almost all the listings were posted in mid of 2024, thus the date adds little info about the price and most sellers that use the estimator would always be using the listing 'today', a date that the model has never seen.
- 'description' : Long text containing details about the real estate brokerage company that is not usefull for the model

In [126]:
df = df.drop(['displayAddress','addedOn','description'], axis = 1)

In [128]:
df.head()

,title,bathrooms,bedrooms,price,verified,furnishing,size_sqft,City,Community
0,Great ROI I High Floor I Creek View,3.0,2.0,2500000,True,No,1323.0,Dubai,Business Bay
1,Full Sea View | Beach Life | Brand New Residence,3.0,2.0,7200000,True,Yes,1410.0,Dubai,Jumeirah Beach Residence
2,Green Belt | Corner Unit | Spacious Plot,3.0,4.0,3600000,True,No,2324.0,Dubai,Villanova
3,2BR+Study | Near Pool and Park | Private,3.0,2.0,2999999,True,No,1647.0,Dubai,The Springs
4,Vacant | Well Maintained | Area Expert,3.0,3.0,2449999,True,No,2105.0,Dubai,Town Square


In [135]:
df[df['bedrooms'].isna()].sample(10)

,title,bathrooms,bedrooms,price,verified,furnishing,size_sqft,City,Community
386,✦Hot Deal| Corner Plot| High ROI| Invest Now,NaN,NaN,3599000,False,No,6135.0,Abu Dhabi,Yas Acres
4485,Unique Payment Plan - Spacious Smart and Central,NaN,NaN,5500000,False,No,2580.0,Dubai,Jumeirah Golf Estates
558,"Residential Plots for sale, townhouse (G+2",NaN,NaN,300000,False,No,1723.0,Ajman,Al Helio
2665,Single Row Mid | Waterfront | Prime Location,NaN,NaN,8700000,False,No,10501.0,Abu Dhabi,Al Qurm
461,"3 years Payment Plan, No Commission Charges",NaN,NaN,41408800,False,No,13993.0,Dubai,Palm Jebel Ali
4929,"Hot Offer The Best Plot 3,153 SQFT G+1+50% Ho...",NaN,NaN,700000,False,No,3153.0,Sharjah,Hoshi
2780,Land for sale in Ajman in Al Helio 2 area,NaN,NaN,300000,False,No,1722.0,Ajman,Al Helio
4268,G+3 Building Plot|Prime Location|Easy Exit to ...,NaN,NaN,1300000,False,No,5381.0,Sharjah,Tilal City
4985,HUGE PLOT I AMAZING LOCATION I LAND INVESTMENT,NaN,NaN,2502662,False,No,10010.0,Dubai,Jebel Ali
2239,"Two Plots Next To Each Other, Best Location, G+5",NaN,NaN,20000000,False,No,28972.0,Dubai,Al Warsan


In [137]:
df[df['bedrooms'].isna()]['price'].describe()

count    1.230000e+02
mean     9.614613e+06
std      1.770877e+07
min      1.250000e+05
25%      7.375000e+05
50%      2.200000e+06
75%      8.850000e+06
max      9.900000e+07
Name: price, dtype: float64

In [138]:
df['price'].describe()

count    5.058000e+03
mean     5.050924e+06
std      1.000928e+07
min      1.000000e+05
25%      1.200000e+06
50%      2.350000e+06
75%      4.499375e+06
max      1.990000e+08
Name: price, dtype: float64

From the comparison where the 'bedroom' values are missing, we can see that the mean and quartile 3 for them have doubled compared to all the listings median and 3rd quartile. Therefore, more values are expensive; therefore, the mean sits at the upper end since it gets affected by the outliers. Therefore, we can assume that these are land plots, whole buildings, or other listings where bedrooms don't apply. If that's the case, we can't fill them since it doesn't apply to that field; thus, we can't fill them with the median values. 

In [140]:
df[df['bedrooms'].isna()].sort_values('price', ascending=False).head(10)[["title", "Community", "size_sqft", "bathrooms", "price"]]

,title,Community,size_sqft,bathrooms,price
427,Luxurious &amp; Elegant | Ready 2025 | Sea View,Jumeirah Bay Island,12651.0,NaN,99000000
1554,Huge Residential Plot | Private Beach Access,Palm Jumeirah,12142.0,NaN,85566000
3978,Prestigious | Beach Access | Freehold,La Mer,26307.0,NaN,80000000
3071,"G+14 Mix use Plot,The Best Location",Al Jaddaf,62803.0,NaN,75363600
431,"Direct Beachfront And Corner Plot, One Of The ...",Jumeirah,12933.0,NaN,52000000
4426,Payment Plan - No Agency Fees - DLD Fees Waiver,Palm Jebel Ali,13993.0,NaN,42618800
461,"3 years Payment Plan, No Commission Charges",Palm Jebel Ali,13993.0,NaN,41408800
4866,Beachfront Plot | Beach Access | Waterfront Li...,Palm Jebel Ali,13993.0,NaN,41408800
4424,Villa Plot - G+2 - Payment Plan - DLD Fees Waiver,Palm Jebel Ali,13993.0,NaN,40428800
2444,Direct Sea View | Single Row | Negotiable,Pearl Jumeirah,9755.0,NaN,40000000


Most of the listings with missing bedrooms are either land plots or full buildings. Since the model estimates the prices of residential units (apartment,villas and townhouses), keeping them would add noise. 

### Exploratory Data Analysis 

In [142]:
df.corr(numeric_only = True)

,bathrooms,bedrooms,price,verified,size_sqft
bathrooms,1.000000,0.931164,0.461971,-0.172192,0.648345
bedrooms,0.931164,1.000000,0.484829,-0.115722,0.667599
price,0.461971,0.484829,1.000000,0.015077,0.609647
verified,-0.172192,-0.115722,0.015077,1.000000,-0.142109
size_sqft,0.648345,0.667599,0.609647,-0.142109,1.000000


- size_sqft has the strongest positive correlation with price (0.609647), followed by bedrooms (0.484829) and then bathrooms(0.461971)  
- bathrooms and bedrooms are really strongly postively correlated (0.931164)

In [144]:
df.head(5)

,title,bathrooms,bedrooms,price,verified,furnishing,size_sqft,City,Community
0,Great ROI I High Floor I Creek View,3.0,2.0,2500000,True,No,1323.0,Dubai,Business Bay
1,Full Sea View | Beach Life | Brand New Residence,3.0,2.0,7200000,True,Yes,1410.0,Dubai,Jumeirah Beach Residence
2,Green Belt | Corner Unit | Spacious Plot,3.0,4.0,3600000,True,No,2324.0,Dubai,Villanova
3,2BR+Study | Near Pool and Park | Private,3.0,2.0,2999999,True,No,1647.0,Dubai,The Springs
4,Vacant | Well Maintained | Area Expert,3.0,3.0,2449999,True,No,2105.0,Dubai,Town Square
